In [2]:
# Before running this notebook, you need to install Playwright and its browser dependencies.
# The following command installs the Playwright library.
!pip install playwright

# This command downloads the necessary browser binaries (Chromium, Firefox, WebKit).
!playwright install


[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## EXPERIMENT 1: Using plain LLM output using search.

Code is generated using ChatGPT with the search function. Website url and prompt are fed to the LLM to get the code snippets.

Uses a few urls in the OSWorld task config suite. Playwright instance is launched, and snapshots are saved to the `/snapshots` dir

In [28]:
from playwright.async_api import async_playwright
import asyncio
import os

urls = ["https://drugs.com", "https://www.lonelyplanet.com", "https://airbnb.com", "https://amazon.com"]
experiment_number = 3
llm_output = """
() => {
  if (document.getElementById("perturbation-cookies-popup")) return;

  const popup = document.createElement("div");
  popup.id = "perturbation-cookies-popup";
  popup.style.position = "fixed";
  popup.style.bottom = "20px";
  popup.style.left = "50%";
  popup.style.transform = "translateX(-50%)";
  popup.style.background = window.getComputedStyle(document.body).backgroundColor || "#fff";
  popup.style.color = window.getComputedStyle(document.body).color || "#000";
  popup.style.fontFamily = window.getComputedStyle(document.body).fontFamily || "sans-serif";
  popup.style.padding = "16px";
  popup.style.borderRadius = "8px";
  popup.style.boxShadow = "0 2px 6px rgba(0,0,0,0.2)";
  popup.style.zIndex = "9999";
  popup.style.maxWidth = "90%";
  popup.style.textAlign = "center";

  const message = document.createElement("div");
  message.textContent = "This website uses cookies to enhance the user experience";
  message.style.marginBottom = "12px";

  const buttonsContainer = document.createElement("div");
  buttonsContainer.style.display = "flex";
  buttonsContainer.style.justifyContent = "center";
  buttonsContainer.style.gap = "8px";

  const makeButton = (label) => {
    const btn = document.createElement("button");
    btn.textContent = label;
    btn.style.fontFamily = "inherit";
    btn.style.fontSize = "inherit";
    btn.style.padding = "8px 12px";
    btn.style.borderRadius = "4px";
    btn.style.border = "1px solid";
    btn.style.cursor = "pointer";
    btn.style.background = "transparent";
    btn.style.color = "inherit";
    btn.addEventListener("click", () => {
      const el = document.getElementById("perturbation-cookies-popup");
      if (el && el.parentNode) el.parentNode.removeChild(el);
    });
    return btn;
  };

  buttonsContainer.appendChild(makeButton("Accept"));
  buttonsContainer.appendChild(makeButton("Decline"));

  popup.appendChild(message);
  popup.appendChild(buttonsContainer);
  document.body.appendChild(popup);
}
"""

async def main():
    async with async_playwright() as p:
        # launch() starts a new browser instance.
        # headless=False means we'll see the UI.
        browser = await p.chromium.launch(headless=True, slow_mo=500)
        
        page = await browser.new_page()
        for url in urls:
            await page.goto(url)

            screenshot_path = os.path.join(f'./snapshots/url_{url.replace("https://", "").replace("www.", "").split(".")[0]}_exp{experiment_number}')
            await page.screenshot(path=screenshot_path + '_before.png', full_page=True)

            await page.evaluate(llm_output)
            await page.screenshot(path=screenshot_path + '_after.png', full_page=True)
        await browser.close()

# In a Jupyter notebook, you can run the async main function like this.
await main()

## EXPERIMENT 2: Use domreducer to compress DOM before passing to LLM

Compress DOM using domreducer and then feed it to LLM.

Uses a few urls in the OSWorld task config suite. Playwright instance is launched, and snapshots are saved to the `/DomreducerSnpashots` dir

In [30]:
!pip install domreducer


[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


## Reducing HTML

Uses domreducer to reduce DOM tree. Reduced HTML outputs are saved to `/reducedHTML` dir

In [ ]:
from playwright.async_api import async_playwright
import asyncio
import os
from domreducer import HtmlReducer

urls = ["https://drugs.com", "https://www.lonelyplanet.com", "https://airbnb.com", "https://amazon.com"]

async def main():
    async with async_playwright() as p:
        # launch() starts a new browser instance.
        # headless=False means we'll see the UI.
        browser = await p.chromium.launch(headless=True, slow_mo=500)
        
        page = await browser.new_page()
        for url in urls:
            await page.goto(url)
            
            # Ensure the directory exists
            directory = './reducedHTMLs/'
            os.makedirs(directory, exist_ok=True)
            path = os.path.join(f'./reducedHTMLs/url_{url.replace("https://", "").replace("www.", "").split(".")[0]}')
            rawHTML = await page.content()
            reducer = HtmlReducer(rawHTML)
            op = reducer.reduce()
            with open(path + '_reduced.html', 'w') as f:
                f.write(op.reduced_data)
            with open(path + '_raw.html', 'w') as f:
                f.write(rawHTML)
            print(f"Site: {url}, Original Size: {len(rawHTML)}, Reduced Size: {len(op.reduced_data)}")

        await browser.close()

# In a Jupyter notebook, you can run the async main function like this.
await main()

Site: https://drugs.com, Original Size: 61876, Reduced Size: 31220
Site: https://www.lonelyplanet.com, Original Size: 471958, Reduced Size: 129574
Site: https://airbnb.com, Original Size: 546315, Reduced Size: 59152
Site: https://amazon.com, Original Size: 1054090, Reduced Size: 198831


In [7]:
from playwright.async_api import async_playwright
import asyncio
import os

urls = ["https://drugs.com", "https://www.lonelyplanet.com", "https://airbnb.com", "https://amazon.com"]
experiment_number = 2
url = urls[0]
llm_output = """
function() {
if (!document.getElementById("perturbation-cookies-popup")) {
var popup = document.createElement("div");
popup.id = "perturbation-cookies-popup";
popup.style.position = "fixed";
popup.style.bottom = "20px";
popup.style.left = "50%";
popup.style.transform = "translateX(-50%)";
popup.style.background = window.getComputedStyle(document.body).backgroundColor || "#fff";
popup.style.color = window.getComputedStyle(document.body).color || "#000";
popup.style.fontFamily = window.getComputedStyle(document.body).fontFamily || "Arial, sans-serif";
popup.style.padding = "15px 25px";
popup.style.borderRadius = "8px";
popup.style.boxShadow = "0 4px 12px rgba(0,0,0,0.15)";
popup.style.zIndex = "99999";
popup.style.display = "flex";
popup.style.alignItems = "center";
popup.style.gap = "15px";

    var message = document.createElement("span");  
    message.textContent = "This website uses cookies to enhance the user experience";  
    popup.appendChild(message);  

    var acceptBtn = document.createElement("button");  
    acceptBtn.textContent = "Accept";  
    acceptBtn.style.padding = "6px 12px";  
    acceptBtn.style.border = "none";  
    acceptBtn.style.borderRadius = "4px";  
    acceptBtn.style.cursor = "pointer";  
    acceptBtn.style.background = "#4CAF50";  
    acceptBtn.style.color = "#fff";  
    acceptBtn.onclick = function() { popup.remove(); };  
    popup.appendChild(acceptBtn);  

    var declineBtn = document.createElement("button");  
    declineBtn.textContent = "Decline";  
    declineBtn.style.padding = "6px 12px";  
    declineBtn.style.border = "none";  
    declineBtn.style.borderRadius = "4px";  
    declineBtn.style.cursor = "pointer";  
    declineBtn.style.background = "#f44336";  
    declineBtn.style.color = "#fff";  
    declineBtn.onclick = function() { popup.remove(); };  
    popup.appendChild(declineBtn);  

    document.body.appendChild(popup);  
}  
"""

async def main():
    async with async_playwright() as p:
        # launch() starts a new browser instance.
        # headless=False means we'll see the UI.
        browser = await p.chromium.launch(headless=True, slow_mo=500)
        
        page = await browser.new_page()

        await page.goto(url)

        screenshot_path = os.path.join(f'./DomReducerSnapshots/url_{url.replace("https://", "").replace("www.", "").split(".")[0]}_exp{experiment_number}')
        await page.screenshot(path=screenshot_path + '_before.png', full_page=True)

        await page.evaluate(llm_output)
        await page.screenshot(path=screenshot_path + '_after.png', full_page=True)
        await browser.close()

# In a Jupyter notebook, you can run the async main function like this.
await main()

Error: Page.evaluate: SyntaxError: Unexpected token ')'
    at eval (<anonymous>)
    at UtilityScript.evaluate (<anonymous>:291:30)
    at UtilityScript.<anonymous> (<anonymous>:1:44)

## EXPERIMENT 3 Domscribe (dom-to-semantic-markdown)

Uses Domscribe library to extract semantic meaning from HTML while preserving some of the tags. Outputs are saved in `/semanticHTMLs`.



In [1]:
!pip install domscribe 
!pip install playwright
!playwright install


[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip available: 22.3.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [3]:
from playwright.async_api import async_playwright
import asyncio
import os
from domscribe import html_to_markdown

options = {
    # 'extract_main_content': True,
    'keep_html': ['div', 'span', 'button', 'a', 'input', 'form'],
    # 'refify_urls': True
}
urls = ["https://drugs.com", "https://www.lonelyplanet.com", "https://airbnb.com", "https://amazon.com"]

async def main():
    async with async_playwright() as p:
        # launch() starts a new browser instance.
        # headless=False means we'll see the UI.
        browser = await p.chromium.launch(headless=True, slow_mo=500)
        
        page = await browser.new_page()
        for url in urls:
            await page.goto(url)
            
            # Ensure the directory exists
            directory = './semanticHTMLs/'
            os.makedirs(directory, exist_ok=True)
            path = os.path.join(f'./semanticHTMLs/url_{url.replace("https://", "").replace("www.", "").split(".")[0]}')
            rawHTML = await page.content()
            markdown_output = html_to_markdown(rawHTML, options)
            with open(path + '_reduced.html', 'w') as f:
                f.write(markdown_output)
            with open(path + '_raw.html', 'w') as f:
                f.write(rawHTML)
            print(f"Site: {url}, Original Size: {len(rawHTML)}, Reduced Size: {len(markdown_output)}")

        await browser.close()

# In a Jupyter notebook, you can run the async main function like this.
await main()

Site: https://drugs.com, Original Size: 61885, Reduced Size: 26768
Site: https://www.lonelyplanet.com, Original Size: 469288, Reduced Size: 113704
Site: https://airbnb.com, Original Size: 544660, Reduced Size: 382341
Site: https://amazon.com, Original Size: 1053996, Reduced Size: 676418
